This notebook generates a mosaic of the DTM files. 

In [1]:
!pip install geopandas shapely rasterio


  Using cached geopandas-1.1.1-py3-none-any.whl (338 kB)
  Using cached shapely-2.1.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (3.1 MB)
  Using cached rasterio-1.4.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (22.2 MB)
  Using cached pyogrio-0.11.1-cp310-cp310-manylinux_2_28_x86_64.whl (27.5 MB)
  Using cached pyproj-3.7.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (9.3 MB)
  Using cached affine-2.4.0-py3-none-any.whl (15 kB)
  Using cached cligj-0.7.2-py3-none-any.whl (7.1 kB)
  Using cached click_plugins-1.1.1.2-py2.py3-none-any.whl (11 kB)


In [2]:
import io
import json
import math
import os
import time

import boto3
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.ndimage import maximum_filter, median_filter, uniform_filter

import geopandas as gpd
from shapely.geometry import box, shape, Polygon, MultiPolygon

import rasterio
from rasterio.crs import CRS
from rasterio.features import shapes
from rasterio.warp import reproject, Resampling
from rasterio.windows import from_bounds

In [3]:
os.environ['S3_ENDPOINT_URL'] = 'http://minio:9000'
os.environ['AWS_ACCESS_KEY_ID'] = 'AKFBKGE170596'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'SKFBKGE200301'

In [4]:
s3 = boto3.client(
    's3',
    endpoint_url=os.environ['S3_ENDPOINT_URL'],
    aws_access_key_id=os.environ['AWS_ACCESS_KEY_ID'],
    aws_secret_access_key=os.environ['AWS_SECRET_ACCESS_KEY']
)

bucket_name = 'dataspace'
path_dtm = 'city-data/dtm/latest/'

In [5]:
import os
import tempfile
import rasterio
from rasterio.merge import merge
from rasterio.io import MemoryFile
import boto3

# S3 client
s3 = boto3.client(
    's3',
    endpoint_url=os.environ['S3_ENDPOINT_URL'],
    aws_access_key_id=os.environ['AWS_ACCESS_KEY_ID'],
    aws_secret_access_key=os.environ['AWS_SECRET_ACCESS_KEY']
)

bucket_name = 'dataspace'
path_dtm = 'city-data/dtm/latest/'
output_path = 'dtm_mosaic.tif'

# Create a temporary directory on disk (avoids using RAM)
with tempfile.TemporaryDirectory() as tmpdir:
    # List all .ASC files in the given path
    response = s3.list_objects_v2(Bucket=bucket_name, Prefix=path_dtm)
    asc_files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.ASC')]

    local_files = []
    for key in asc_files:
        local_path = os.path.join(tmpdir, os.path.basename(key))
        # Download each file to the temp folder
        s3.download_file(bucket_name, key, local_path)
        local_files.append(local_path)

    # Open all rasters as datasets (but not all loaded in RAM)
    src_files_to_mosaic = [rasterio.open(fp) for fp in local_files]

    # Merge rasters on disk
    mosaic, out_trans = merge(src_files_to_mosaic)

    # Copy metadata from the first raster
    out_meta = src_files_to_mosaic[0].meta.copy()
    out_meta.update({
        "driver": "GTiff",
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": out_trans
    })

    # Write the final mosaic to disk (not in memory)
    with rasterio.open(output_path, "w", **out_meta) as dest:
        dest.write(mosaic)

    # Close datasets
    for src in src_files_to_mosaic:
        src.close()

print(f"Mosaico guardado en: {output_path}")



Mosaico guardado en: dtm_mosaic.tif
